In [0]:
from pyspark.sql.functions import sha2, concat_ws, col

def add_hash_column(df, business_cols):
    return df.withColumn(
        "hash_key",
        sha2(concat_ws("||", *[col(c) for c in business_cols]), 256)
    )

In [0]:
def get_incremental_data(bronze_table, table_name):
    last_date = spark.sql(f"""
        SELECT last_processed_date 
        FROM control.config_batch 
        WHERE table_name = '{table_name}'
    """).collect()[0][0]

    df = spark.table(bronze_table) \
        .filter(col("batch_date") > last_date)

    return df

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp, lit

def scd2_merge(source_df, target_table, business_key):

    delta_target = DeltaTable.forName(spark, target_table)

    merge_condition = f"t.{business_key} = s.{business_key} AND t.is_current = true"

    (
        delta_target.alias("t")
        .merge(
            source_df.alias("s"),
            merge_condition
        )
        .whenMatchedUpdate(
            condition="t.hash_key <> s.hash_key",
            set={
                "effective_to": "current_timestamp()",
                "is_current": "false"
            }
        )
        .whenNotMatchedInsert(
            values={
                business_key: f"s.{business_key}",
                "hash_key": "s.hash_key",
                "effective_from": "current_timestamp()",
                "effective_to": "null",
                "is_current": "true"
            }
        )
        .execute()
    )

In [0]:
def scd1_merge(source_df, target_table, business_key):

    delta_target = DeltaTable.forName(spark, target_table)

    merge_condition = f"t.{business_key} = s.{business_key}"

    (
        delta_target.alias("t")
        .merge(source_df.alias("s"), merge_condition)
        .whenMatchedUpdateAll(condition="t.hash_key <> s.hash_key")
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
def insert_fact_history(source_df, target_table, history_table, business_key):

    target_df = spark.table(target_table)

    changed_df = target_df.alias("t") \
        .join(source_df.alias("s"),
              f"t.{business_key} = s.{business_key}") \
        .filter("t.hash_key <> s.hash_key")

    changed_df.write.format("delta") \
        .mode("append") \
        .saveAsTable(history_table)

In [0]:
def process_dim(table_name, bronze_table, silver_table, business_key, business_cols):

    df = get_incremental_data(bronze_table, table_name)

    df = add_hash_column(df, business_cols)

    scd2_merge(df, silver_table, business_key)

In [0]:
def process_fact(table_name, bronze_table, silver_table, history_table, business_key, business_cols):

    df = get_incremental_data(bronze_table, table_name)

    df = add_hash_column(df, business_cols)

    insert_fact_history(df, silver_table, history_table, business_key)

    scd1_merge(df, silver_table, business_key)